# Visualization Layer Preparation - Gold Layer

Prepares visualization-ready tables for the Streamlit app.

**Inputs:**
- `{catalog}.{gold_schema}.expansion_candidates_h3_enhanced`
- `{catalog}.{silver_schema}.existing_stores_h3`
- `{catalog}.{silver_schema}.pois_competitors`
- `{catalog}.{silver_schema}.isochrones_convenience`
- `{catalog}.{bronze_schema}.census_states`

**Outputs:**
- `{catalog}.{gold_schema}.viz_h3_grid` - H3-8 covering MA boundary (authoritative boundary filter)
- `{catalog}.{gold_schema}.viz_expansion_candidates` - Enhanced with distance, proximity, fulfillment strategy
- `{catalog}.{gold_schema}.viz_existing_stores` - Store info with H3 cell ID
- `{catalog}.{gold_schema}.viz_competitors` - Competitor POIs with H3 cell ID
- `{catalog}.{gold_schema}.viz_convenience` - Convenience isochrones with candidate proximity
- `{catalog}.{gold_schema}.viz_network_metrics` - Singleton aggregates for dashboard
- `{catalog}.{gold_schema}.viz_optimization_results` - Pre-computed optimization results

**Key Optimizations:**
- H3 cell membership for MA boundary filtering (replaces lat/long bounding box)
- Pre-computed Haversine distances (replaces O(n²) runtime calculation)
- Pre-computed convenience proximity (replaces runtime point-in-polygon)
- Pre-computed fulfillment strategy (partner vs new_store)
- Pre-computed optimization results for parameter grid

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, explode, lit, when
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", "jdub_demo_aws")
dbutils.widgets.text("bronze_schema", "geo_bronze")
dbutils.widgets.text("silver_schema", "geo_silver")
dbutils.widgets.text("gold_schema", "geo_gold")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")

print(f"Catalog: {catalog}")
print(f"Bronze: {bronze_schema}, Silver: {silver_schema}, Gold: {gold_schema}")

## 1. Generate H3 Grid Covering Massachusetts

In [ ]:
# Load Massachusetts boundary
ma_boundary = spark.table(f"{catalog}.{bronze_schema}.census_states").filter(
    (col("state_abbr") == "MA") | (col("state_fips") == "25")
)

# Generate H3-8 grid covering Massachusetts
viz_h3_grid = ma_boundary.select(
    explode(expr("h3_coverash3string(ST_AsBinary(geometry), 5)")).alias("coarse_h3")
).select(
    explode(expr("h3_tochildren(coarse_h3, 8)")).alias("h3_cell_id")
).distinct().withColumn(
    "geometry", expr("ST_GeomFromWKT(h3_boundaryaswkt(h3_cell_id), 4326)")
).withColumn(
    "center_lat", expr("ST_Y(ST_GeomFromWKT(h3_centeraswkt(h3_cell_id), 4326))")
).withColumn(
    "center_lon", expr("ST_X(ST_GeomFromWKT(h3_centeraswkt(h3_cell_id), 4326))")
)

print(f"Generated H3-8 grid with {viz_h3_grid.count()} cells")

# Write H3 grid
viz_h3_grid.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_h3_grid")
print(f"Written to {catalog}.{gold_schema}.viz_h3_grid")

## 2. Prepare Expansion Candidates (Enhanced with Distance, Proximity, Fulfillment Strategy)

This is the most enhanced table - it includes:
- Normalized scores (0-1) for visualization
- Pre-computed minimum distance to existing stores (replaces O(n²) runtime Haversine)
- Pre-computed convenience store proximity (replaces runtime point-in-polygon)
- Pre-computed fulfillment strategy recommendation (partner vs new_store)
- Quality tier for quick filtering
- H3 cell membership filter for MA boundary (replaces lat/long bounding box)

In [ ]:
# Load expansion candidates and filter to MA using H3 grid (authoritative boundary)
candidates_raw = spark.table(f"{catalog}.{gold_schema}.expansion_candidates_h3_enhanced")
h3_grid = spark.table(f"{catalog}.{gold_schema}.viz_h3_grid").select("h3_cell_id")

# Filter candidates to only those within MA boundary using H3 membership
candidates = candidates_raw.join(h3_grid, "h3_cell_id", "inner")
print(f"Candidates filtered to MA via H3 grid: {candidates.count()} (from {candidates_raw.count()})")

# Load existing stores for distance calculation
existing_stores = spark.table(f"{catalog}.{silver_schema}.existing_stores_h3").select(
    col("store_number"),
    col("latitude").alias("store_lat"),
    col("longitude").alias("store_lon")
)

# Load convenience isochrones for proximity check
# Note: isochrones_convenience uses store_type (e.g., "7-Eleven") not a separate name column
convenience_isochrones = spark.table(f"{catalog}.{silver_schema}.isochrones_convenience").select(
    col("location_id").alias("convenience_id"),
    col("store_type").alias("conv_store_type"),
    col("city").alias("conv_city"),
    col("store_type").alias("conv_store_name"),  # Use store_type as the name (e.g., "7-Eleven")
    col("drive_time_minutes").alias("conv_drive_time"),
    "geometry"
)

print(f"Existing stores for distance calc: {existing_stores.count()}")
print(f"Convenience isochrones for proximity: {convenience_isochrones.count()}")

### 2.1 Pre-compute Distance to Existing Stores (Haversine)

This replaces the O(n²) runtime distance calculation in the app's optimization loop.
For each candidate, we calculate the minimum distance to any existing store using the Haversine formula.

In [ ]:
# Cross join candidates with existing stores and calculate Haversine distance
# Haversine formula: d = 2r * arcsin(sqrt(sin²((lat2-lat1)/2) + cos(lat1)*cos(lat2)*sin²((lon2-lon1)/2)))
# Using 3959 miles as Earth's radius

candidates_x_stores = candidates.crossJoin(existing_stores)

# Calculate distance in miles using Haversine formula
candidates_with_distances = candidates_x_stores.withColumn(
    "distance_miles",
    expr("""
        3959 * 2 * asin(sqrt(
            power(sin(radians(store_lat - latitude) / 2), 2) +
            cos(radians(latitude)) * cos(radians(store_lat)) *
            power(sin(radians(store_lon - longitude) / 2), 2)
        ))
    """)
)

# Get minimum distance and nearest store for each candidate
from pyspark.sql.window import Window

window_spec = Window.partitionBy("h3_cell_id").orderBy("distance_miles")

candidate_distances = candidates_with_distances.withColumn(
    "row_num", F.row_number().over(window_spec)
).filter(col("row_num") == 1).select(
    "h3_cell_id",
    col("distance_miles").alias("min_distance_to_existing"),
    col("store_number").alias("nearest_existing_store")
)

print(f"Calculated distances for {candidate_distances.count()} candidates")
print("\nDistance distribution (miles):")
candidate_distances.select("min_distance_to_existing").summary().show()

### 2.2 Pre-compute Convenience Store Proximity (Point-in-Polygon)

This replaces the ~75 lines of JavaScript ray-casting point-in-polygon code in the app.
For each candidate, we check if it falls within any convenience store's 5-minute drive isochrone using ST_CONTAINS.

In [ ]:
# Check which candidates fall within convenience store isochrones using ST_CONTAINS
# This is a spatial join - candidate point within isochrone polygon

# Create point geometry for candidates with SRID 4326 to match isochrone geometry
candidates_with_point = candidates.withColumn(
    "candidate_point", expr("ST_SetSRID(ST_Point(longitude, latitude), 4326)")
)

# Spatial join: find candidates within convenience isochrones
candidates_in_convenience = candidates_with_point.join(
    convenience_isochrones,
    expr("ST_Contains(geometry, candidate_point)"),
    "left"
)

# For candidates in multiple isochrones, keep the one with shortest drive time
window_conv = Window.partitionBy("h3_cell_id").orderBy("conv_drive_time")

convenience_proximity = candidates_in_convenience.withColumn(
    "row_num", F.row_number().over(window_conv)
).filter(col("row_num") == 1).select(
    "h3_cell_id",
    when(col("convenience_id").isNotNull(), True).otherwise(False).alias("within_convenience_isochrone"),
    col("convenience_id"),
    col("conv_store_type").alias("convenience_store_type"),
    col("conv_city").alias("convenience_city"),
    col("conv_store_name").alias("convenience_store_name"),
    col("conv_drive_time").alias("convenience_drive_time")
)

in_isochrone_count = convenience_proximity.filter(col("within_convenience_isochrone") == True).count()
print(f"Candidates within convenience isochrones: {in_isochrone_count}")
print(f"Candidates outside convenience isochrones: {convenience_proximity.count() - in_isochrone_count}")

### 2.3 Assemble Enhanced Expansion Candidates Table

Join all pre-computed columns and add:
- Normalized scores (0-1)
- Quality tier for quick filtering
- Fulfillment strategy (partner vs new_store)
- GeoJSON geometry for map rendering
- Center lat/lon for markers

In [ ]:
# Calculate min/max for normalization
stats = candidates.agg(
    F.min("predicted_annual_sales").alias("min_sales"),
    F.max("predicted_annual_sales").alias("max_sales"),
    F.min("population").alias("min_pop"),
    F.max("population").alias("max_pop")
).collect()[0]

min_sales, max_sales = stats["min_sales"], stats["max_sales"]
min_pop, max_pop = stats["min_pop"], stats["max_pop"]

# Join candidates with distance and convenience proximity data
viz_candidates = candidates.join(
    candidate_distances, "h3_cell_id", "left"
).join(
    convenience_proximity, "h3_cell_id", "left"
)

# Add city column: use convenience_city if available, otherwise derive from urbanicity
viz_candidates = viz_candidates.withColumn(
    "city",
    when(col("convenience_city").isNotNull(), col("convenience_city"))
    .otherwise(
        when(col("urbanicity") == "urban", "Boston Metro")
        .when(col("urbanicity") == "suburban", "Greater Boston")
        .otherwise("Massachusetts")
    )
)

# Add normalized scores
if max_sales > min_sales:
    viz_candidates = viz_candidates.withColumn(
        "normalized_sales_score",
        (col("predicted_annual_sales") - lit(min_sales)) / lit(max_sales - min_sales)
    )
else:
    viz_candidates = viz_candidates.withColumn("normalized_sales_score", lit(0.5))

if max_pop > min_pop:
    viz_candidates = viz_candidates.withColumn(
        "normalized_pop_score",
        (col("population") - lit(min_pop)) / lit(max_pop - min_pop)
    )
else:
    viz_candidates = viz_candidates.withColumn("normalized_pop_score", lit(0.5))

# Add percentile rank, quality tier, fulfillment strategy, and geometry columns
viz_candidates = viz_candidates.withColumn(
    "percentile_rank",
    F.percent_rank().over(Window.orderBy("predicted_annual_sales"))
).withColumn(
    "quality_tier",
    when(col("percentile_rank") >= 0.75, "top_25")
    .when(col("percentile_rank") >= 0.50, "top_50")
    .when(col("percentile_rank") >= 0.25, "top_75")
    .otherwise("bottom_25")
).withColumn(
    "fulfillment_strategy",
    when(col("within_convenience_isochrone") == True, "partner").otherwise("new_store")
).withColumn(
    "geometry", expr("ST_GeomFromWKT(h3_boundaryaswkt(h3_cell_id), 4326)")
).withColumn(
    "geometry_geojson", expr("ST_AsGeoJSON(ST_GeomFromWKT(h3_boundaryaswkt(h3_cell_id), 4326))")
).withColumn(
    "center_lat", expr("ST_Y(ST_GeomFromWKT(h3_centeraswkt(h3_cell_id), 4326))")
).withColumn(
    "center_lon", expr("ST_X(ST_GeomFromWKT(h3_centeraswkt(h3_cell_id), 4326))")
)

# Fill nulls for distance and convenience columns
viz_candidates = viz_candidates.fillna({
    "min_distance_to_existing": 999.0,
    "within_convenience_isochrone": False
})

print(f"Enhanced candidates with {viz_candidates.count()} rows")
print("\nColumn list:")
print(viz_candidates.columns)

# Write enhanced table
viz_candidates.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_expansion_candidates")
print(f"\nWritten to {catalog}.{gold_schema}.viz_expansion_candidates")

## 3. Prepare Existing Stores

In [ ]:
try:
    existing_stores = spark.table(f"{catalog}.{silver_schema}.existing_stores_h3")
    
    # Get column list to handle optional columns
    existing_cols = existing_stores.columns
    
    viz_existing = existing_stores.select(
        "store_number",
        "latitude",
        "longitude",
        "store_type" if "store_type" in existing_cols else lit("LCE").alias("store_type"),
        "city",
        "state",
        "population",
        col("total_poi_count").alias("poi_count") if "total_poi_count" in existing_cols else lit(0).alias("poi_count"),
        "geometry"
    ).withColumn(
        "marker_type", lit("existing_lce")
    ).withColumn(
        # Add H3 cell ID for consistent filtering with viz_h3_grid
        # Note: h3_longlatash3string takes (longitude, latitude, resolution)
        "h3_cell_id", expr("h3_longlatash3string(longitude, latitude, 8)")
    ).withColumn(
        "geometry_geojson", expr("ST_AsGeoJSON(geometry)")
    )
    
    print(f"Prepared {viz_existing.count()} existing stores")
    
    viz_existing.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_existing_stores")
    print(f"Written to {catalog}.{gold_schema}.viz_existing_stores")
    
except Exception as e:
    print(f"Could not prepare existing stores: {e}")

## 4. Prepare Competitors

In [ ]:
try:
    competitors = spark.table(f"{catalog}.{silver_schema}.pois_competitors")
    
    viz_competitors = competitors.select(
        col("poi_id").alias("id"),
        "name",
        "latitude",
        "longitude",
        "poi_category",
        "poi_subcategory",
        "address"
    ).withColumn(
        "marker_type", lit("competitor")
    ).withColumn(
        "geometry", expr("ST_Point(longitude, latitude)")
    ).withColumn(
        # Add H3 cell ID for consistent filtering with viz_h3_grid
        # Note: h3_longlatash3string takes (longitude, latitude, resolution)
        "h3_cell_id", expr("h3_longlatash3string(longitude, latitude, 8)")
    ).withColumn(
        "geometry_geojson", expr("ST_AsGeoJSON(ST_Point(longitude, latitude))")
    )
    
    print(f"Prepared {viz_competitors.count()} competitors")
    
    viz_competitors.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_competitors")
    print(f"Written to {catalog}.{gold_schema}.viz_competitors")
    
except Exception as e:
    print(f"Could not prepare competitors: {e}")

## 5. Prepare Convenience Store Isochrones (Enhanced with Candidate Proximity)

This table now includes:
- Count of expansion candidates within each convenience isochrone
- Total predicted sales of candidates in isochrone (partnership potential)
- Array of candidate H3 cell IDs within isochrone for reverse lookup

In [ ]:
try:
    convenience = spark.table(f"{catalog}.{silver_schema}.isochrones_convenience")
    candidates_for_conv = spark.table(f"{catalog}.{gold_schema}.expansion_candidates_h3_enhanced")
    
    # Create point geometry for candidates with SRID 4326 to match isochrone geometry
    candidates_with_point = candidates_for_conv.withColumn(
        "candidate_point", expr("ST_SetSRID(ST_Point(longitude, latitude), 4326)")
    ).select("h3_cell_id", "candidate_point", "predicted_annual_sales")
    
    # Join convenience isochrones with candidates that fall within them
    convenience_with_candidates = convenience.alias("c").join(
        candidates_with_point.alias("cand"),
        expr("ST_Contains(c.geometry, cand.candidate_point)"),
        "left"
    )
    
    # Aggregate candidate info per convenience store
    candidate_agg = convenience_with_candidates.groupBy(
        col("c.location_id")
    ).agg(
        F.count("cand.h3_cell_id").alias("candidate_count_in_isochrone"),
        F.sum("cand.predicted_annual_sales").alias("total_candidate_sales_in_isochrone"),
        F.collect_list("cand.h3_cell_id").alias("candidate_h3_cells_in_isochrone")
    )
    
    # Join back to convenience base table (use aliases to avoid ambiguity)
    # Note: isochrones_convenience uses store_type (e.g., "7-Eleven") not a separate name column
    viz_convenience = convenience.alias("conv").join(
        candidate_agg.alias("agg"),
        col("conv.location_id") == col("agg.location_id"),
        "left"
    ).select(
        col("conv.location_id").alias("id"),
        col("conv.store_type"),
        col("conv.store_type").alias("name"),  # Use store_type as name (e.g., "7-Eleven")
        col("conv.latitude"),
        col("conv.longitude"),
        col("conv.city"),
        col("conv.state"),
        col("conv.drive_time_minutes"),
        col("conv.area_sqkm"),
        col("conv.geometry"),
        F.coalesce(col("agg.candidate_count_in_isochrone"), lit(0)).alias("candidate_count_in_isochrone"),
        F.coalesce(col("agg.total_candidate_sales_in_isochrone"), lit(0)).alias("total_candidate_sales_in_isochrone"),
        F.coalesce(col("agg.candidate_h3_cells_in_isochrone"), F.array()).alias("candidate_h3_cells_in_isochrone")
    ).withColumn(
        "marker_type", lit("convenience")
    ).withColumn(
        "geometry_geojson", expr("ST_AsGeoJSON(geometry)")
    ).withColumn(
        # Note: h3_longlatash3string takes (longitude, latitude, resolution)
        "h3_cell_id", expr("h3_longlatash3string(longitude, latitude, 8)")
    )
    
    print(f"Prepared {viz_convenience.count()} convenience store isochrones")
    
    # Show partnership potential summary
    print("\nPartnership potential summary:")
    viz_convenience.select(
        "candidate_count_in_isochrone", "total_candidate_sales_in_isochrone"
    ).summary().show()
    
    viz_convenience.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_convenience")
    print(f"Written to {catalog}.{gold_schema}.viz_convenience")
    
except Exception as e:
    print(f"Could not prepare convenience isochrones: {e}")
    import traceback
    traceback.print_exc()

## 6. Create Network Metrics (Singleton Aggregates)

Pre-computed aggregate metrics for dashboard display. This eliminates runtime aggregation loops in the app.

In [ ]:
from datetime import datetime

try:
    # Load data for metrics calculation
    existing = spark.table(f"{catalog}.{silver_schema}.existing_stores_h3")
    candidates = spark.table(f"{catalog}.{gold_schema}.expansion_candidates_h3_enhanced")
    
    # Calculate total POI count for existing stores
    existing_with_poi = existing.withColumn(
        "total_poi_count",
        F.coalesce(col("total_retail_pois"), lit(0)) +
        F.coalesce(col("total_food_drink_pois"), lit(0)) +
        F.coalesce(col("total_leisure_pois"), lit(0)) +
        F.coalesce(col("total_education_pois"), lit(0)) +
        F.coalesce(col("total_healthcare_pois"), lit(0)) +
        F.coalesce(col("total_financial_pois"), lit(0)) +
        F.coalesce(col("total_tourism_pois"), lit(0)) +
        F.coalesce(col("total_transportation_pois"), lit(0))
    )
    
    # Calculate existing store metrics
    existing_metrics = existing_with_poi.agg(
        F.count("*").alias("total_existing_stores"),
        F.avg("population").alias("avg_store_population"),
        F.avg("total_poi_count").alias("avg_store_poi"),
        F.sum("population").alias("total_network_population")
    ).collect()[0]
    
    # Calculate candidate metrics
    candidate_metrics = candidates.agg(
        F.count("*").alias("total_candidates"),
        F.avg("predicted_annual_sales").alias("avg_candidate_sales"),
        F.avg("population").alias("avg_candidate_population"),
        F.percentile_approx("predicted_annual_sales", 0.25).alias("sales_p25"),
        F.percentile_approx("predicted_annual_sales", 0.50).alias("sales_p50"),
        F.percentile_approx("predicted_annual_sales", 0.75).alias("sales_p75"),
        F.min("predicted_annual_sales").alias("sales_min"),
        F.max("predicted_annual_sales").alias("sales_max")
    ).collect()[0]
    
    # Create singleton metrics row (use Python datetime instead of F.current_timestamp())
    viz_network_metrics = spark.createDataFrame([{
        "total_existing_stores": int(existing_metrics["total_existing_stores"]),
        "avg_store_population": float(existing_metrics["avg_store_population"] or 0),
        "avg_store_poi": float(existing_metrics["avg_store_poi"] or 0),
        "total_network_population": int(existing_metrics["total_network_population"] or 0),
        "total_candidates": int(candidate_metrics["total_candidates"]),
        "avg_candidate_sales": float(candidate_metrics["avg_candidate_sales"] or 0),
        "avg_candidate_population": float(candidate_metrics["avg_candidate_population"] or 0),
        "sales_p25": float(candidate_metrics["sales_p25"] or 0),
        "sales_p50": float(candidate_metrics["sales_p50"] or 0),
        "sales_p75": float(candidate_metrics["sales_p75"] or 0),
        "sales_min": float(candidate_metrics["sales_min"] or 0),
        "sales_max": float(candidate_metrics["sales_max"] or 0),
        "last_updated": datetime.now()
    }])
    
    print("Network Metrics Summary:")
    print(f"  Existing stores: {existing_metrics['total_existing_stores']}")
    print(f"  Avg store population: {existing_metrics['avg_store_population']:,.0f}")
    print(f"  Total candidates: {candidate_metrics['total_candidates']}")
    print(f"  Avg candidate sales: ${candidate_metrics['avg_candidate_sales']:,.0f}")
    print(f"  Sales percentiles (25/50/75): ${candidate_metrics['sales_p25']:,.0f} / ${candidate_metrics['sales_p50']:,.0f} / ${candidate_metrics['sales_p75']:,.0f}")
    
    viz_network_metrics.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_network_metrics")
    print(f"\nWritten to {catalog}.{gold_schema}.viz_network_metrics")
    
except Exception as e:
    print(f"Could not create network metrics: {e}")
    import traceback
    traceback.print_exc()

## 7. Pre-compute Optimization Results

This is the highest-impact optimization. Instead of running O(n²) optimization at runtime in the app, we pre-compute results for common parameter combinations.

**Parameter Grid:**
- max_stores: [5, 10, 15, 20, 25, 30]
- min_distance_new: [1.0, 2.0, 3.0, 5.0]
- min_distance_existing: [1.0, 2.0, 3.0, 5.0]

Total combinations: 6 × 4 × 4 = 96

**Impact:** Optimization latency goes from 500-2000ms to <50ms (O(1) lookup)

In [ ]:
from itertools import product
import math

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate distance in miles between two points using Haversine formula."""
    R = 3959  # Earth's radius in miles
    
    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    
    a = math.sin(dlat/2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    
    return R * c

def run_greedy_optimization(candidates_pdf, existing_stores_pdf, max_stores, min_dist_new, min_dist_existing):
    """
    Greedy optimization algorithm that selects candidates based on:
    1. Highest predicted sales first
    2. Must be >= min_dist_existing from all existing stores
    3. Must be >= min_dist_new from all already-selected candidates
    """
    # Sort by predicted sales descending
    sorted_candidates = candidates_pdf.sort_values('predicted_annual_sales', ascending=False)
    
    selected = []
    selected_coords = []
    
    for _, candidate in sorted_candidates.iterrows():
        if len(selected) >= max_stores:
            break
            
        cand_lat = candidate['latitude']
        cand_lon = candidate['longitude']
        
        # Check distance to existing stores
        too_close_to_existing = False
        for _, store in existing_stores_pdf.iterrows():
            dist = haversine_distance(cand_lat, cand_lon, store['latitude'], store['longitude'])
            if dist < min_dist_existing:
                too_close_to_existing = True
                break
        
        if too_close_to_existing:
            continue
            
        # Check distance to already-selected candidates
        too_close_to_selected = False
        for sel_lat, sel_lon in selected_coords:
            dist = haversine_distance(cand_lat, cand_lon, sel_lat, sel_lon)
            if dist < min_dist_new:
                too_close_to_selected = True
                break
        
        if too_close_to_selected:
            continue
            
        # Candidate passes all checks
        selected.append(candidate['h3_cell_id'])
        selected_coords.append((cand_lat, cand_lon))
    
    return selected

# Parameter grid
max_stores_options = [5, 10, 15, 20, 25, 30]
min_dist_new_options = [1.0, 2.0, 3.0, 5.0]
min_dist_existing_options = [1.0, 2.0, 3.0, 5.0]

print(f"Parameter grid: {len(max_stores_options)} × {len(min_dist_new_options)} × {len(min_dist_existing_options)} = {len(max_stores_options) * len(min_dist_new_options) * len(min_dist_existing_options)} combinations")

# Load data as pandas for efficient iteration
candidates_pdf = spark.table(f"{catalog}.{gold_schema}.viz_expansion_candidates").select(
    "h3_cell_id", "latitude", "longitude", "predicted_annual_sales"
).toPandas()

existing_stores_pdf = spark.table(f"{catalog}.{silver_schema}.existing_stores_h3").select(
    "store_number", "latitude", "longitude"
).toPandas()

print(f"Loaded {len(candidates_pdf)} candidates and {len(existing_stores_pdf)} existing stores")

In [ ]:
# Run optimization for all parameter combinations
from datetime import datetime

results = []
total_combinations = len(max_stores_options) * len(min_dist_new_options) * len(min_dist_existing_options)
current = 0

print("Running optimization for all parameter combinations...")

for max_stores, min_dist_new, min_dist_existing in product(max_stores_options, min_dist_new_options, min_dist_existing_options):
    current += 1
    
    # Run greedy optimization
    selected_h3_cells = run_greedy_optimization(
        candidates_pdf, existing_stores_pdf, 
        max_stores, min_dist_new, min_dist_existing
    )
    
    # Calculate total predicted sales for selected candidates
    selected_sales = candidates_pdf[candidates_pdf['h3_cell_id'].isin(selected_h3_cells)]['predicted_annual_sales'].sum()
    
    results.append({
        'max_stores': max_stores,
        'min_distance_new': min_dist_new,
        'min_distance_existing': min_dist_existing,
        'selected_h3_cells': selected_h3_cells,
        'selected_count': len(selected_h3_cells),
        'total_predicted_sales': float(selected_sales),
        'computed_at': datetime.now()
    })
    
    if current % 20 == 0:
        print(f"  Progress: {current}/{total_combinations} ({100*current/total_combinations:.0f}%)")

print(f"\nCompleted {len(results)} optimization results")

# Show sample results
print("\nSample results:")
for r in results[:5]:
    print(f"  max={r['max_stores']}, dist_new={r['min_distance_new']}, dist_exist={r['min_distance_existing']} -> {r['selected_count']} stores, ${r['total_predicted_sales']:,.0f}")

In [ ]:
# Convert results to Spark DataFrame and write to Delta
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, ArrayType, StringType, TimestampType

schema = StructType([
    StructField("max_stores", IntegerType(), False),
    StructField("min_distance_new", DoubleType(), False),
    StructField("min_distance_existing", DoubleType(), False),
    StructField("selected_h3_cells", ArrayType(StringType()), False),
    StructField("selected_count", IntegerType(), False),
    StructField("total_predicted_sales", DoubleType(), False),
    StructField("computed_at", TimestampType(), False)
])

optimization_results_df = spark.createDataFrame(results, schema)

print(f"Writing {optimization_results_df.count()} optimization results to Delta...")

optimization_results_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.viz_optimization_results")

print(f"Written to {catalog}.{gold_schema}.viz_optimization_results")

# Show statistics
print("\nOptimization results statistics:")
optimization_results_df.select("selected_count", "total_predicted_sales").summary().show()

## Summary

In [ ]:
print("=" * 60)
print("VISUALIZATION LAYER SUMMARY")
print("=" * 60)

viz_tables = [
    "viz_h3_grid",
    "viz_expansion_candidates",
    "viz_existing_stores",
    "viz_competitors",
    "viz_convenience",
    "viz_network_metrics",
    "viz_optimization_results"
]

for table_name in viz_tables:
    try:
        count = spark.table(f"{catalog}.{gold_schema}.{table_name}").count()
        print(f"  {table_name}: {count:,} rows")
    except Exception as e:
        print(f"  {table_name}: NOT AVAILABLE - {e}")

print("\n" + "=" * 60)
print("VISUALIZATION LAYER COMPLETE")
print("=" * 60)

# Show enhanced columns in viz_expansion_candidates
print("\nEnhanced viz_expansion_candidates columns:")
enhanced_cols = ["min_distance_to_existing", "nearest_existing_store", 
                 "within_convenience_isochrone", "convenience_store_name",
                 "fulfillment_strategy", "quality_tier", "geometry_geojson"]
for col_name in enhanced_cols:
    print(f"  ✓ {col_name}")

## Folium Map Visualization with Marker Clustering

Interactive map showing:
- **LCE Stores** (green markers with clustering)
- **LCE Trade Areas** (orange isochrones)
- **Convenience Store Trade Areas** (blue isochrones)
- **Expansion Candidates** (gold/orange markers)

Uses Little Caesars branding colors and marker clustering for performance.

In [ ]:
%pip install folium --quiet

import folium
from folium.plugins import MarkerCluster, Fullscreen
import pandas as pd
from shapely import wkt
from shapely.geometry import mapping

print("Loading data for Folium map...")

# Load LCE stores
lce_stores_pd = (
    spark.table(f"{catalog}.{gold_schema}.viz_existing_stores")
    .select("store_number", "latitude", "longitude", "city", "state", "population")
    .toPandas()
)

# Load LCE isochrones as GeoJSON
lce_isochrones_df = spark.table(f"{catalog}.{silver_schema}.isochrones_lce")
lce_isochrones_gdf = (
    lce_isochrones_df
    .selectExpr(
        "location_id as store_number",
        "ST_AsText(geometry) as geometry_wkt",
        "drive_time_minutes",
        "area_sqkm"
    )
    .toPandas()
)

lce_isochrones_geojson = {
    "type": "FeatureCollection",
    "features": []
}

for _, row in lce_isochrones_gdf.iterrows():
    geom = wkt.loads(row['geometry_wkt'])
    feature = {
        "type": "Feature",
        "properties": {
            "store_number": row['store_number'],
            "drive_time_minutes": row['drive_time_minutes'],
            "area_sqkm": row['area_sqkm']
        },
        "geometry": mapping(geom)
    }
    lce_isochrones_geojson["features"].append(feature)

# Load convenience store isochrones as GeoJSON
convenience_isochrones_df = spark.table(f"{catalog}.{silver_schema}.isochrones_convenience")
convenience_isochrones_gdf = (
    convenience_isochrones_df
    .selectExpr(
        "location_id",
        "ST_AsText(geometry) as geometry_wkt",
        "drive_time_minutes",
        "area_sqkm"
    )
    .toPandas()
)

convenience_isochrones_geojson = {
    "type": "FeatureCollection",
    "features": []
}

for _, row in convenience_isochrones_gdf.iterrows():
    geom = wkt.loads(row['geometry_wkt'])
    feature = {
        "type": "Feature",
        "properties": {
            "location_id": row['location_id'],
            "drive_time_minutes": row['drive_time_minutes'],
            "area_sqkm": row['area_sqkm']
        },
        "geometry": mapping(geom)
    }
    convenience_isochrones_geojson["features"].append(feature)

# Load expansion candidates
candidates_pd = (
    spark.table(f"{catalog}.{gold_schema}.viz_expansion_candidates")
    .orderBy(F.desc("predicted_annual_sales"))
    .limit(20)  # Top 20 candidates
    .select("h3_cell_id", "latitude", "longitude", "predicted_annual_sales", "population")
    .toPandas()
)

print(f"\nLoaded data:")
print(f"  LCE stores: {len(lce_stores_pd)}")
print(f"  LCE trade areas: {len(lce_isochrones_geojson['features'])}")
print(f"  Convenience trade areas: {len(convenience_isochrones_geojson['features'])}")
print(f"  Expansion candidates: {len(candidates_pd)}")

In [ ]:
# Create Folium map centered on Massachusetts
ma_center = [42.4072, -71.3824]
folium_map = folium.Map(
    location=ma_center,
    zoom_start=9,
    tiles='https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png',
    attr='CartoDB'
)

print("Building map layers...")

# Layer 1: LCE Trade Areas (orange isochrones)
if lce_isochrones_geojson['features']:
    for feature in lce_isochrones_geojson['features']:
        coords = feature['geometry']['coordinates'][0]
        polygon_coords = [[lat, lon] for lon, lat in coords]
        
        folium.Polygon(
            locations=polygon_coords,
            color='#FF8C00',  # Orange stroke
            fill=True,
            fillColor='#FF8C00',
            fillOpacity=0.1,
            weight=1,
            popup=f"<b>LCE Store {feature['properties']['store_number']}</b><br>"
                  f"Drive time: {feature['properties']['drive_time_minutes']} min<br>"
                  f"Area: {feature['properties']['area_sqkm']:.2f} km²"
        ).add_to(folium_map)
    
    print(f"  Added {len(lce_isochrones_geojson['features'])} LCE trade areas")

# Layer 2: Convenience Trade Areas (blue isochrones)
if convenience_isochrones_geojson['features']:
    for feature in convenience_isochrones_geojson['features']:
        coords = feature['geometry']['coordinates'][0]
        polygon_coords = [[lat, lon] for lon, lat in coords]
        
        folium.Polygon(
            locations=polygon_coords,
            color='#3b82f6',  # Blue stroke
            fill=True,
            fillColor='#3b82f6',
            fillOpacity=0.08,
            weight=1,
            popup=f"<b>Convenience Store</b><br>"
                  f"Location: {feature['properties']['location_id']}<br>"
                  f"Drive time: {feature['properties']['drive_time_minutes']} min<br>"
                  f"Area: {feature['properties']['area_sqkm']:.2f} km²"
        ).add_to(folium_map)
    
    print(f"  Added {len(convenience_isochrones_geojson['features'])} convenience trade areas")

# Layer 3: LCE Store Markers with GREEN clustering
lce_cluster = MarkerCluster(
    name='LCE Stores',
    options={
        'maxClusterRadius': 50,
        'iconCreateFunction': '''
            function(cluster) {
                return L.divIcon({
                    html: '<div style="background-color: #10b981; color: white; border-radius: 50%; width: 40px; height: 40px; display: flex; align-items: center; justify-content: center; font-weight: bold; border: 3px solid #059669;"><span>' + cluster.getChildCount() + '</span></div>',
                    className: 'marker-cluster',
                    iconSize: L.point(40, 40)
                });
            }
        '''
    }
).add_to(folium_map)

for _, store in lce_stores_pd.iterrows():
    folium.CircleMarker(
        location=[store['latitude'], store['longitude']],
        radius=8,
        popup=f"<b>Little Caesars</b><br>"
              f"Store: {store['store_number']}<br>"
              f"Location: {store.get('city', 'N/A')}, {store.get('state', 'N/A')}<br>"
              f"Population: {store.get('population', 0):,.0f}",
        tooltip=f"LCE Store {store['store_number']}",
        color='#10b981',
        fill=True,
        fillColor='#34d399',
        fillOpacity=0.8,
        weight=2
    ).add_to(lce_cluster)

print(f"  Added {len(lce_stores_pd)} LCE store markers with clustering")

# Layer 4: Expansion Candidates (gold/orange markers)
for _, candidate in candidates_pd.iterrows():
    folium.CircleMarker(
        location=[candidate['latitude'], candidate['longitude']],
        radius=8,
        popup=f"<b>Expansion Candidate</b><br>"
              f"H3 Cell: {candidate['h3_cell_id']}<br>"
              f"Predicted Sales: ${candidate['predicted_annual_sales']:,.0f}<br>"
              f"Population: {candidate['population']:,.0f}",
        tooltip=f"Expansion: ${candidate['predicted_annual_sales']:,.0f}",
        color='#f59e0b',
        fill=True,
        fillColor='#fbbf24',
        fillOpacity=0.8,
        weight=2
    ).add_to(folium_map)

print(f"  Added {len(candidates_pd)} expansion candidate markers")

print("\nMap layers complete!")

In [ ]:
# Add legend
legend_html = '''
<div style="position: fixed; 
            bottom: 50px; right: 50px; width: 300px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:14px; padding: 15px; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.3);">
<p style="margin-bottom: 12px; font-weight: bold; font-size: 16px; color: #1a1a1a;">LCE Hunger Detection Map</p>
<p style="margin: 8px 0;">
    <span style="display: inline-block; width: 20px; height: 20px; 
                 background-color: #10b981; border-radius: 50%; border: 2px solid #059669;
                 vertical-align: middle; margin-right: 8px;"></span>
    <strong>LCE Stores</strong> (clustered)
</p>
<p style="margin: 8px 0;">
    <span style="display: inline-block; width: 40px; height: 12px; 
                 background-color: rgba(255,140,0,0.1); 
                 border: 2px solid #FF8C00; 
                 vertical-align: middle; margin-right: 8px;"></span>
    LCE Trade Areas (5-min)
</p>
<p style="margin: 8px 0;">
    <span style="display: inline-block; width: 40px; height: 12px; 
                 background-color: rgba(59,130,246,0.08); 
                 border: 2px solid #3b82f6; 
                 vertical-align: middle; margin-right: 8px;"></span>
    Convenience Trade Areas
</p>
<p style="margin: 8px 0;">
    <span style="display: inline-block; width: 20px; height: 20px; 
                 background-color: #fbbf24; border-radius: 50%; border: 2px solid #f59e0b;
                 vertical-align: middle; margin-right: 8px;"></span>
    Expansion Candidates
</p>
<p style="margin-top: 12px; padding-top: 8px; border-top: 1px solid #ddd; font-size: 11px; color: #666;">
    <strong>Tip:</strong> Zoom in/out to see clusters split/merge
</p>
</div>
'''

folium_map.get_root().html.add_child(folium.Element(legend_html))

# Add fullscreen button
Fullscreen(
    position='topleft',
    title='Expand map',
    title_cancel='Exit fullscreen',
    force_separate_button=True
).add_to(folium_map)

# Display the map
folium_map